# PBI Package Tutorial

This comprehensive tutorial demonstrates how to use the PBI (Phage-Bacteria Interaction) Python package for analyzing phage-host interaction data.## What You'll Learn1. **Database Connection**: Connect to the PBI database and explore its contents2. **Data Retrieval**: Query and filter phage-host interaction data3. **Sequence Access**: Retrieve phage and host genome sequences4. **Log Analysis**: Understand tracking and diagnostic information5. **Streaming Datasets**: Use memory-efficient datasets for machine learning6. **Architecture**: Understand how the components work together## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                    PBI Architecture
│
├─────────────────────────────────────────────────────────┤
│
│
│  ┌──────────────┐     ┌─────────────┐                   │
│  │  DuckDB      │───▶│ Metadata    │                   │
│  │  Database    │     │ Queries     │                   │
│  └──────────────┘     └─────────────┘                   │
│         │                                               │
│         │                                               │
│  ┌──────▼──────────────────────────────────────┐        │
│  │     SequenceRetriever                       │        │
│  │  - Lazy FASTA loading                       │        │
│  │  - Index-based sequence access              │        │
│  │  - Query + sequence integration             │        │ 
│  └──────┬──────────────────────────────────────┘        │
│         │                                               │
│         ├─────────────┬──────────────┬─────────────┐    │
│         │             │              │             │    │
│  ┌──────▼──────┐ ┌───▼────┐  ┌──────▼──────┐ ┌───▼──┐   │
│  │Indexed FASTA│ │Host    │  │Streaming    │ │Index-│   │
│  │- Phages     │ │Mapping │  │Dataset      │ │ed    │   │
│  │- Proteins   │ │JSON    │  │(ML)         │ │Data  │   │
│  └─────────────┘ └────────┘  └─────────────┘ └──────┘   │
│                                                         │
└─────────────────────────────────────────────────────────┘
```
**Key Components:**
- **DuckDB Database**: Stores metadata (taxonomy, assembly info, etc.)
- **Indexed FASTA Files**: Store sequences with `.fai` index for fast random access
- **SequenceRetriever**: Main interface combining metadata queries with sequence retrieval
- **Host Mapping JSON**: Maps Host_ID to individual genome FASTA files
- **Datasets**: PyTorch-compatible datasets for machine learning workflows

---

## 1. Installation and Setup

First, let's ensure the PBI package is properly installed and import necessary libraries.

In [1]:
# Import PBI and other useful libraries
import pbi
import pandas as pd
import time
from pathlib import Path

# Set pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ PBI package imported successfully")
print(f"   Version: {pbi.__version__ if hasattr(pbi, '__version__') else 'dev'}")

✅ PBI package imported successfully
   Version: 0.1.0


---

## 2. Database Connection and Exploration

The PBI package provides a convenient `quick_connect()` function that automatically finds and connects to your database.

### How Connection Works:

1. **Auto-detection**: Searches common locations for database and FASTA files
2. **Lazy Loading**: Database connects immediately, but FASTA files load in background
3. **Ready to Query**: Can query database while sequences are being indexed

This design allows you to start working immediately without waiting for large FASTA files to load.

In [2]:
# Quick connect (auto-detects database and FASTA files)
print("🔍 Connecting to PBI database...")
start = time.time()
retriever = pbi.quick_connect()
elapsed = time.time() - start

print(f"✅ Connected in {elapsed:.2f} seconds")
print(f"   Database: {retriever.conn}")
print(f"   Phage FASTA: {'Ready' if retriever._phage_fasta else 'Loading...'}")
print(f"   Host data: {'Available' if retriever._has_host_data else 'Not loaded'}")

2026-02-20 07:29:52,692 - INFO - 📂 Checking FASTA index files:
2026-02-20 07:29:52,694 - INFO -    Phage index: True (52570.4 KB)
2026-02-20 07:29:52,694 - INFO -    Protein index: True (1432185.2 KB)
2026-02-20 07:29:52,695 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-02-20 07:29:52,701 - INFO -    Loaded mapping for 5525 hosts
2026-02-20 07:29:52,701 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-02-20 07:29:52,733 - INFO - 🔄 Starting background FASTA loading...
2026-02-20 07:29:52,734 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-02-20 07:29:52,734 - INFO - ✅ Initialization complete (FASTA loading in background)


🔍 Connecting to PBI database...
✅ Connected in 0.04 seconds
   Database: <_duckdb.DuckDBPyConnection object at 0x7fee0febefb0>
   Phage FASTA: Loading...
   Host data: Available


### Database Statistics

Let's explore what's in the database:

In [3]:
# Get comprehensive statistics
stats = retriever.get_stats()

print("📊 Database Statistics:")
print("=" * 60)
print(f"Phages:   {stats['database']['phages']:,}")
print(f"Proteins: {stats['database']['proteins']:,}")
print(f"Hosts:    {stats['database']['hosts']:,}")
print(f"Phage-Host Associations: {stats['database']['phage_host_associations']:,}")
print()
print("FASTA Files:")
print(f"  Phage sequences:   {stats['fasta']['phages']:,}")
print(f"  Protein sequences: {stats['fasta']['proteins']:,}")

2026-02-20 07:29:52,745 - INFO - ⏳ Waiting for FASTA loading to complete...
2026-02-20 07:30:09,397 - INFO -    ✅ Phage FASTA loaded in 16.66s (873,717 sequences)
2026-02-20 07:30:09,399 - INFO - 🔄 [Background] Loading protein FASTA: /data/processed/sequences/all_proteins.fasta
2026-02-20 07:34:52,752 - WARNING - ⚠️ Timeout after 300s - FASTA may still be loading
2026-02-20 07:36:32,700 - INFO -    ✅ Protein FASTA loaded in 383.30s (31,050,116 sequences)
2026-02-20 07:36:32,703 - INFO -    ℹ️  Using on-demand loading for 5,525 individual host files
2026-02-20 07:36:32,705 - INFO - 🎉 All FASTA files loaded in 399.97s
2026-02-20 07:36:41,138 - INFO - 🔍 Sample phage keys:
2026-02-20 07:36:41,140 - INFO -    - 'AE002163.1...'
2026-02-20 07:36:41,141 - INFO -    - 'AF009630.1...'
2026-02-20 07:36:41,142 - INFO -    - 'AF011378.1...'
2026-02-20 07:36:41,144 - INFO - 🔍 Sample protein keys:
2026-02-20 07:36:41,144 - INFO -    - 'AE002163.1 AAF39720.1...'
2026-02-20 07:36:41,145 - INFO -    - '

📊 Database Statistics:
Phages:   873,718
Proteins: 43,088,582
Hosts:    5,538
Phage-Host Associations: 782,089

FASTA Files:
  Phage sequences:   873,717
  Protein sequences: 31,050,116


---

## 3. Data Retrieval and Filtering

Now let's query the database to retrieve specific data. We'll explore different filtering options.

### Understanding the Data Model

The database uses a **star schema** with fact and dimension tables:

- **fact_phages**: Main phage information (length, GC content, taxonomy)
- **dim_hosts**: Host bacterial information (species, assembly, genome stats)
- **dim_proteins**: Protein sequences and annotations
- **phage_host_associations**: Links phages to their hosts

You can filter on any column using SQL WHERE clause syntax.

In [4]:
# Example 1: Get complete phages with high GC content
print("Example 1: Complete phages")
print("-" * 60)

phage_metadata = retriever.get_phage_metadata(
    #where_clause="Completeness = 'complete'",
    where_clause="",
    limit=100000
)

print(f"Found {len(phage_metadata)} phages")
print(phage_metadata[['Phage_ID', 'Length', 'GC_content', 'Taxonomy', 'Completeness']])
print(set(phage_metadata['Completeness']))

2026-02-20 07:36:41,257 - INFO - 🔍 Querying phage metadata...


Example 1: Complete phages
------------------------------------------------------------


2026-02-20 07:36:41,467 - INFO - ✅ Retrieved metadata for 100,000 phages


Found 100000 phages
          Phage_ID  Length  GC_content      Taxonomy    Completeness
0      NC_001330.1    6087   45.178249  Microviridae    High-quality
1      NC_001331.1    7349   61.491359    Inoviridae     Low-quality
2      NC_001332.1    6744   42.719454    Inoviridae  Medium-quality
3      NC_001335.1   52297   62.257873  Caudovirales    High-quality
4      NC_001341.1    4491   33.288800          None  Not-determined
...            ...     ...         ...           ...             ...
99995  uvig_213191   21260   54.012230  Caudovirales  Medium-quality
99996  uvig_213194   19098   52.817049  Caudovirales     Low-quality
99997  uvig_213201   15178   43.279747          None  Medium-quality
99998  uvig_213204   14469   55.802060  Caudovirales     Low-quality
99999  uvig_213205   14410   55.010409  Caudovirales     Low-quality

[100000 rows x 5 columns]
{'Medium-quality', 'Not-determined', 'Complete', 'Low-quality', 'High-quality'}


In [5]:
# diagnostic
import pandas as pd

# Check the source CSV
csv_path = "/data/intermediate/csv/merged/merged_phage_metadata.csv"
df = pd.read_csv(csv_path)

print(f"Total phages: {len(df):,}")
print(f"\nCompleteness column info:")
print(df['Completeness'].value_counts(dropna=False))
print(f"\nNull/NaN count: {df['Completeness'].isna().sum():,}")
print(f"\nSample values:")
print(df[['Phage_ID', 'Completeness', 'Source_DB']].head(20))

Total phages: 873,718

Completeness column info:
Completeness
High-quality      300137
Low-quality       267050
Medium-quality    212175
Complete           72668
Not-determined     21688
Name: count, dtype: int64

Null/NaN count: 0

Sample values:
       Phage_ID    Completeness Source_DB
0   NC_001330.1    High-quality    RefSeq
1   NC_001331.1     Low-quality    RefSeq
2   NC_001332.1  Medium-quality    RefSeq
3   NC_001335.1    High-quality    RefSeq
4   NC_001341.1  Not-determined    RefSeq
5   NC_001365.1    High-quality    RefSeq
6   NC_001396.1    High-quality    RefSeq
7   NC_002014.1  Medium-quality    RefSeq
8   NC_001416.1    High-quality    RefSeq
9   NC_001418.1    High-quality    RefSeq
10  NC_001422.1  Medium-quality    RefSeq
11  NC_001423.1    High-quality    RefSeq
12  NC_001426.1    High-quality    RefSeq
13  NC_001447.1  Not-determined    RefSeq
14  NC_001604.1        Complete    RefSeq
15  NC_001609.1    High-quality    RefSeq
16  NC_001628.1    High-quality    Ref

In [6]:
import pandas as pd
import glob
from pathlib import Path

# Path to TSV files in Docker
tsv_dir = Path("/data/intermediate/csv/phage_metadata/")

# Find all TSV files
tsv_files = list(tsv_dir.glob("*.tsv"))

print(f"✅ Found {len(tsv_files)} TSV files\n")

# Check each file
for tsv_file in sorted(tsv_files):
    print("=" * 80)
    print(f"📄 File: {tsv_file.name}")
    print("=" * 80)
    
    try:
        # Read just first 100 rows for speed
        df = pd.read_csv(tsv_file, sep='\t', nrows=100, low_memory=False)
        
        print(f"   Total columns: {len(df.columns)}")
        print(f"   Sample rows: {len(df):,}")
        
        # Check if Completeness column exists
        if 'Completeness' in df.columns:
            print("   ✅ Completeness column EXISTS!")
            
            # Check values
            completeness_values = df['Completeness'].value_counts(dropna=False)
            print(f"\n   Completeness value distribution:")
            print(completeness_values)
            
            # Check for nulls
            null_count = df['Completeness'].isna().sum()
            total = len(df)
            print(f"\n   Null/NaN: {null_count:,} / {total:,} ({null_count/total*100:.1f}%)")
            
            # Show sample non-null values
            non_null = df[df['Completeness'].notna()]
            if len(non_null) > 0:
                print(f"\n   Sample non-null values:")
                print(non_null[['Phage_ID', 'Completeness']].head(5))
        else:
            print("   ❌ Completeness column NOT FOUND!")
            print(f"\n   First 10 columns: {list(df.columns)[:10]}")
        
        print()
        
    except Exception as e:
        print(f"   ❌ Error: {e}\n")

✅ Found 14 TSV files

📄 File: CHVD_Phage_Metadata_URL.tsv
   Total columns: 10
   Sample rows: 100
   ✅ Completeness column EXISTS!

   Completeness value distribution:
Completeness
Low-quality       48
High-quality      24
Medium-quality    22
Not-determined     4
Complete           2
Name: count, dtype: int64

   Null/NaN: 0 / 100 (0.0%)

   Sample non-null values:
                       Phage_ID    Completeness
0     SRS101376_a1_ct99314_vs01    High-quality
1     SRS101388_a1_ct42327_vs01    High-quality
2     SRS101376_a1_ct26874_vs01     Low-quality
3       SAMEA1906416_a1_ct13001    High-quality
4  SAMEA1906416_a1_ct130485_vs1  Medium-quality

📄 File: DDBJ_Phage_Metadata_URL.tsv
   Total columns: 9
   Sample rows: 100
   ✅ Completeness column EXISTS!

   Completeness value distribution:
Completeness
High-quality      60
Complete          34
Low-quality        4
Medium-quality     2
Name: count, dtype: int64

   Null/NaN: 0 / 100 (0.0%)

   Sample non-null values:
     Phage_ID  

In [7]:
# Example 2: Get hosts with complete genomes
print("\nExample 2: Hosts with complete genome assemblies")
print("-" * 60)

host_metadata = retriever.get_host_metadata(
    where_clause="Assembly_Level = 'Complete Genome'",
    limit=10
)

print(f"Found {len(host_metadata)} hosts")
print(host_metadata[['Species_Name', 'Assembly_Level', 'Genome_Length', 'GC_Content']].head())

2026-02-20 07:36:46,027 - INFO - 🔍 Querying host metadata...
2026-02-20 07:36:46,032 - INFO - ✅ Retrieved metadata for 10 hosts



Example 2: Hosts with complete genome assemblies
------------------------------------------------------------
Found 10 hosts
                Species_Name   Assembly_Level  Genome_Length  GC_Content
0   Lawsonia intracellularis  Complete Genome        1699035       32.94
1  Mycobacteroides abscessus  Complete Genome        4989215       64.23
2       Pseudomonas syringae  Complete Genome        6012225       59.02
3       Enterobacter cloacae  Complete Genome        4935288       54.92
4     Pseudomonas aeruginosa  Complete Genome        6417544       66.41


In [8]:
# Example 3: Complex filter - Phage-host pairs with specific criteria
print("\nExample 3: Large phages (>100kb) infecting Escherichia")
print("-" * 60)

phage_host_data = retriever.get_phage_host_metadata(
    where_clause="Length > 100000 AND Species_Name LIKE 'Escherichia%'",
    limit=10
)

print(f"Found {len(phage_host_data)} interactions")
print(phage_host_data[['Phage_ID', 'Phage_Length', 'Host_Species', 'Host_Assembly_Level']].head())

2026-02-20 07:36:46,044 - INFO - 🔍 Querying phage-host metadata...



Example 3: Large phages (>100kb) infecting Escherichia
------------------------------------------------------------


2026-02-20 07:36:46,179 - INFO - ✅ Retrieved metadata for 10 phage-host pairs


Found 10 interactions
             Phage_ID  Phage_Length      Host_Species Host_Assembly_Level
0  MGV-GENOME-0374074        135180  Escherichia coli     Complete Genome
1  MGV-GENOME-0370890        108927  Escherichia coli     Complete Genome
2  MGV-GENOME-4434933        108696  Escherichia coli     Complete Genome
3         uvig_327287        140563  Escherichia coli     Complete Genome
4         uvig_374229        167569  Escherichia coli     Complete Genome


### Using LIMIT and OFFSET

The `where_clause` parameter now supports `LIMIT` and `OFFSET` directly:

In [9]:
# Example 4: Pagination with LIMIT and OFFSET
print("Example 4: Pagination - Get 2nd batch of 5 phages")
print("-" * 60)

# First 5 phages
batch1 = retriever.get_phage_metadata(where_clause="LIMIT 5")
print(f"Batch 1: {len(batch1)} phages")
print(batch1['Phage_ID'].tolist())

# Next 5 phages (using OFFSET)
batch2 = retriever.get_phage_metadata(where_clause="LIMIT 5 OFFSET 5")
print(f"\nBatch 2: {len(batch2)} phages")
print(batch2['Phage_ID'].tolist())

# Combined filter + limit
batch3 = retriever.get_phage_metadata(
    where_clause="Completeness = 'complete' LIMIT 5"
)
print(f"\nBatch 3 (filtered): {len(batch3)} complete phages")
print(batch3['Phage_ID'].tolist())

2026-02-20 07:36:46,192 - INFO - 🔍 Querying phage metadata...


Example 4: Pagination - Get 2nd batch of 5 phages
------------------------------------------------------------


2026-02-20 07:36:46,197 - INFO - ✅ Retrieved metadata for 5 phages
2026-02-20 07:36:46,198 - INFO - 🔍 Querying phage metadata...
2026-02-20 07:36:46,203 - INFO - ✅ Retrieved metadata for 5 phages
2026-02-20 07:36:46,204 - INFO - 🔍 Querying phage metadata...
2026-02-20 07:36:46,207 - INFO - ✅ Retrieved metadata for 0 phages


Batch 1: 5 phages
['NC_001330.1', 'NC_001331.1', 'NC_001332.1', 'NC_001335.1', 'NC_001341.1']

Batch 2: 5 phages
['NC_001365.1', 'NC_001396.1', 'NC_002014.1', 'NC_001416.1', 'NC_001418.1']

Batch 3 (filtered): 0 complete phages
[]


---

## 4. Sequence Retrieval (Non-Streaming)

Now let's retrieve actual DNA sequences. The `get_phage_host_pairs()` method combines metadata with sequences.

### How Sequence Retrieval Works:

1. **Query Database**: Get metadata matching your filter
2. **Fetch from FASTA**: Use `.fai` index to quickly locate sequences
3. **Combine**: Return DataFrame with both metadata and sequences

This is **non-streaming** - all data is loaded into memory. Good for small to medium datasets (<10,000 pairs).

In [10]:
# Example 5: Get phage-host pairs with sequences
print("Example 5: Retrieve phage-host pairs with sequences")
print("-" * 60)

# Get 5 pairs
pairs = retriever.get_phage_host_pairs(
    where_clause="LIMIT 5"
)

print(f"Retrieved {len(pairs)} phage-host pairs with sequences")
print("\nMetadata columns:", list(pairs.columns))
print("\nSample data:")
print(pairs[['Phage_ID', 'Species_Name', 'Phage_Length', 'Host_Length']].head())

# Check sequence content
print("\nSequence preview:")
for idx, row in pairs.head(2).iterrows():
    print(f"  Phage {row['Phage_ID'][:30]}: {row['Phage_Sequence'][:50]}...")
    print(f"  Host {row['Species_Name'][:30]}: {row['Host_Sequence'][:50]}...")
    print()

2026-02-20 07:36:46,218 - INFO - 🔍 Querying phage-host pairs...


Example 5: Retrieve phage-host pairs with sequences
------------------------------------------------------------


2026-02-20 07:36:46,669 - INFO - 📊 Found 5 phage-host pairs
2026-02-20 07:36:46,672 - INFO - 📥 Fetching sequences for 5 phages and 4 unique hosts
2026-02-20 07:36:46,750 - INFO - ✅ Retrieved 5 complete phage-host pairs with sequences


Retrieved 5 phage-host pairs with sequences

Metadata columns: ['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence']

Sample data:
                                            Phage_ID  \
0  Station23_DCM_ALL_assembly_NODE_1538_length_13...   
1  Station25_DCM_ALL_assembly_NODE_1185_length_19...   
2  Station25_DCM_ALL_assembly_NODE_1676_length_15...   
3  Station25_DCM_ALL_assembly_NODE_1714_length_15...   
4  Station30_DCM_ALL_assembly_NODE_985_length_188...   

               Species_Name  Phage_Length  Host_Length  
0  Streptococcus agalactiae         13963      2034123  
1    Streptococcus pyogenes         19508      1897626  
2  Clostridioides difficile         15647      4168273  
3    Streptococcus pyogenes         15448      1897626  
4   Prochloro

In [11]:
# Example 6: Filter by taxonomy
print("Example 6: Get Siphoviridae phages and their hosts")
print("-" * 60)

sipho_pairs = retriever.get_phage_host_pairs(
    where_clause="Taxonomy LIKE '%Siphoviridae%' LIMIT 10"
)

print(f"Found {len(sipho_pairs)} Siphoviridae phage-host pairs")

if len(sipho_pairs) > 0:
    print("\nTaxonomy distribution:")
    print(sipho_pairs['Phage_Taxonomy'].value_counts())

2026-02-20 07:36:46,775 - INFO - 🔍 Querying phage-host pairs...


Example 6: Get Siphoviridae phages and their hosts
------------------------------------------------------------


2026-02-20 07:36:47,095 - INFO - 📊 Found 10 phage-host pairs
2026-02-20 07:36:47,097 - INFO - 📥 Fetching sequences for 10 phages and 7 unique hosts
2026-02-20 07:36:47,212 - INFO - ✅ Retrieved 10 complete phage-host pairs with sequences


Found 10 Siphoviridae phage-host pairs

Taxonomy distribution:
Phage_Taxonomy
Siphoviridae    10
Name: count, dtype: int64


---

## 5. Log Analysis and Tracking Information

Understanding where to find diagnostic information is crucial for troubleshooting and quality control.

### Types of Tracking Information:

1. **Host Download Metadata** (`host_metadata.csv`) - Successful downloads
2. **Failure Logs** (`failed_downloads.txt`) - What couldn't be downloaded
3. **Missing Hosts CSV** (from datasets) - Runtime missing data tracking
4. **Snakemake Logs** - Pipeline execution logs

Let's examine each:

In [32]:
# Example 7: Analyze host download metadata
print("Example 7: Host Download Metadata Analysis")
print("-" * 60)

# Path to metadata (adjust based on your setup)
metadata_path = Path("/data/intermediate/csv/merged/host_metadata.csv")

if metadata_path.exists():
    host_meta = pd.read_csv(metadata_path)
    
    # Convert GC_Content to mumeric
    host_meta['GC_Content'] = pd.to_numeric(host_meta['GC_Content'], errors='coerce')
    host_meta['Genome_Length'] = pd.to_numeric(host_meta['Genome_Length'], errors='coerce')
    
    
    print(f"Total host genomes downloaded: {len(host_meta)}")
    print(f"\nAssembly level distribution:")
    print(host_meta['Assembly_Level'].value_counts())
    
    #print(f"\nSequence count statistics:")
    #print(f"  Mean: {host_meta['Sequence_Count'].mean():.1f}")
    #print(f"  Median: {host_meta['Sequence_Count'].median():.0f}")
    #print(f"  Max: {host_meta['Sequence_Count'].max():.0f}")
    
    # Identify highly fragmented genomes
    #fragmented = host_meta[host_meta['Sequence_Count'] > 100]
    #print(f"\nHighly fragmented genomes (>100 sequences): {len(fragmented)}")
    #if len(fragmented) > 0:
    #    print(fragmented[['Species_Name', 'Assembly_Level', 'Sequence_Count']].head())
    
    # GC content distribution
    print(f"\nGC content range: {host_meta['GC_Content'].min():.1f}% - {host_meta['GC_Content'].max():.1f}%")

    print(host_meta.describe())
    
else:
    print(f"⚠️  Metadata file not found at: {metadata_path}")
    print("   Run the host genome download pipeline first.")
    print("   See: docs/guides/pipeline-execution.md")

Example 7: Host Download Metadata Analysis
------------------------------------------------------------
Total host genomes downloaded: 5538

Assembly level distribution:
Assembly_Level
Complete Genome    2093
Scaffold           1730
Contig             1597
Chromosome          118
Name: count, dtype: int64

GC content range: 15.3% - 74.8%
       Genome_Length   GC_Content
count   5.525000e+03  5525.000000
mean    6.632267e+06    52.766757
std     8.295427e+07    12.523793
min     4.129000e+03    15.260000
25%     2.940356e+06    41.840000
50%     4.131279e+06    53.380000
75%     5.346738e+06    64.010000
max     3.298431e+09    74.800000


In [33]:
# Example 8: Check for common issues
print("\nExample 8: Quality Control Checks")
print("-" * 60)

if metadata_path.exists():
    
    # Check 1: Very small genomes (possible issues)
    small = host_meta[host_meta['Genome_Length'] < 1000000]
    print(f"Small genomes (<1 Mbp): {len(small)}")
    if len(small) > 0:
        print("  These may be plasmids or incomplete downloads")
        print(small[['Species_Name', 'Genome_Length', 'Assembly_Level']].head())
    
    # Check 2: Extreme GC content (possible contamination)
    extreme_gc = host_meta[(host_meta['GC_Content'] < 30) | (host_meta['GC_Content'] > 75)]
    print(f"\nExtreme GC content (<30% or >75%): {len(extreme_gc)}")
    if len(extreme_gc) > 0:
        print(extreme_gc[['Species_Name', 'GC_Content', 'Assembly_Level']].head())
    
    # Check 3: Recent downloads
    host_meta['Download_Date'] = pd.to_datetime(host_meta['Download_Date'])
    recent = host_meta[host_meta['Download_Date'] > pd.Timestamp.now() - pd.Timedelta(days=30)]
    print(f"\nGenomes downloaded in last 30 days: {len(recent)}")
else:
    print("Run Example 7 first to load metadata")


Example 8: Quality Control Checks
------------------------------------------------------------
Small genomes (<1 Mbp): 37
  These may be plasmids or incomplete downloads
                              Species_Name  Genome_Length   Assembly_Level
124             Metamycoplasma arthritidis       806209.0  Complete Genome
130                Mycoplasmopsis pulmonis       962791.0  Complete Genome
384     Candidatus Southlakia epibionticum       864133.0  Complete Genome
2594          Candidatus Carsonella ruddii       166875.0  Complete Genome
3834  Candidatus Nanosyncoccus nanoralicus       563736.0         Scaffold

Extreme GC content (<30% or >75%): 135
                                       Species_Name  GC_Content  \
20                          Fusobacterium nucleatum       27.14   
37                 Candidatus Pelagibacter communis       28.92   
46                         Clostridioides difficile       28.51   
100                      Intestinibacter bartlettii       29.01   
126 

### Reading Failure Logs

When genomes fail to download, understanding why helps improve data completeness:

In [34]:
# Example 9: Analyze failure log
print("Example 9: Download Failure Analysis")
print("-" * 60)

failure_log = Path("data/logs/failed_downloads.txt")

if failure_log.exists():
    with open(failure_log) as f:
        failures = f.read()
    
    # Count failures by category
    import re
    categories = re.findall(r'Category: (.*?) \((\d+)', failures)
    
    print("Failure breakdown:")
    for category, count in categories:
        print(f"  {category}: {count}")
    
    # Show sample failures
    print("\nSample failed species:")
    species_matches = re.findall(r'^  - (.+)$', failures, re.MULTILINE)
    for species in species_matches[:5]:
        print(f"  • {species}")
else:
    print(f"⚠️  Failure log not found at: {failure_log}")
    print("   This is normal if all downloads succeeded!")

Example 9: Download Failure Analysis
------------------------------------------------------------
⚠️  Failure log not found at: data/logs/failed_downloads.txt
   This is normal if all downloads succeeded!


### Runtime Missing Hosts Tracking

When using datasets, you can track which phage-host pairs have missing hosts:

In [15]:
# Example 10: Track missing hosts during dataset usage
print("Example 10: Runtime Missing Hosts Tracking")
print("-" * 60)

# Create a dataset with missing hosts tracking
missing_csv = Path("/tmp/missing_hosts_example.csv")

try:
    dataset = retriever.create_indexed_dataset(
        where_clause="LIMIT 100",
        missing_hosts_csv=str(missing_csv)
    )
    
    print(f"Created dataset with {len(dataset)} samples")
    
    # Iterate through a few samples (this triggers host lookup)
    for i, sample in enumerate(dataset):
        if i >= 10:  # Just check first 10
            break
    
    # Check if any hosts were missing
    if missing_csv.exists():
        missing = pd.read_csv(missing_csv)
        print(f"\nMissing hosts found: {len(missing)}")
        
        if len(missing) > 0:
            print("\nFailure reasons:")
            print(missing['Failure_Reason'].value_counts())
            print("\nSample missing hosts:")
            print(missing[['Phage_ID', 'Species_Name', 'Failure_Reason']].head())
    else:
        print("✅ All hosts were successfully retrieved!")
    
except Exception as e:
    print(f"⚠️  Could not create dataset: {e}")
    print("   Make sure host genomes are downloaded and indexed.")

2026-02-20 07:37:55,550 - INFO - Using host mapping with 5525 hosts


Example 10: Runtime Missing Hosts Tracking
------------------------------------------------------------


2026-02-20 07:37:56,025 - INFO - Loaded metadata for 100 phage-host pairs


Created dataset with 100 samples
✅ All hosts were successfully retrieved!


---

## 6. Streaming Datasets for Machine Learning

For large-scale machine learning, loading all data into memory isn't feasible. **Streaming datasets** solve this by loading data on-demand.

### Streaming vs. Indexed Datasets:

| Feature | Indexed Dataset | Streaming Dataset |
|---------|----------------|-------------------|
| Memory Usage | High (all metadata in RAM) | Low (batches only) |
| Random Access | Yes (shuffling supported) | No (sequential) |
| Use Case | Medium datasets (<100k pairs) | Large datasets (>100k pairs) |
| PyTorch DataLoader | ✅ Full support | ✅ Full support |

### How Streaming Works:

```
Database Query (batched)
    ↓
Fetch batch of metadata
    ↓
Load sequences for batch
    ↓
Yield samples one by one
    ↓
Repeat until exhausted
```

This keeps memory constant regardless of dataset size.

In [16]:
# Example 11: Create and use streaming dataset
print("Example 11: Streaming Dataset")
print("-" * 60)

# Create streaming dataset
stream_dataset = retriever.create_streaming_dataset(
    where_clause="LIMIT 50",  # Small for demo
    batch_size=10  # Fetch 10 at a time from DB
)

print(f"Created streaming dataset")
print(f"  Batch size: 10")
print(f"  Total (estimated): ~50 samples")

# Iterate through dataset
print("\nStreaming samples:")
for i, sample in enumerate(stream_dataset):
    if i >= 5:  # Show first 5
        break
    print(f"  Sample {i+1}:")
    print(f"    Phage: {sample['Phage_ID'][:40]}")
    print(f"    Host: {sample['Species_Name'][:40]}")
    print(f"    Phage length: {sample['Phage_Length']:,} bp")
    print(f"    Has sequence: {len(sample['Phage_Sequence']) > 0}")
    print()

print("✅ Streaming completed")

2026-02-20 07:38:26,636 - INFO - Using host mapping with 5525 hosts


Example 11: Streaming Dataset
------------------------------------------------------------
Created streaming dataset
  Batch size: 10
  Total (estimated): ~50 samples

Streaming samples:
  Sample 1:
    Phage: Station25_DCM_ALL_assembly_NODE_1673_len
    Host: Segatella copri
    Phage length: 15,681 bp
    Has sequence: True

  Sample 2:
    Phage: Station25_DCM_ALL_assembly_NODE_1732_len
    Host: Vibrio parahaemolyticus
    Phage length: 15,347 bp
    Has sequence: True

  Sample 3:
    Phage: Station25_DCM_ALL_assembly_NODE_2606_len
    Host: Enterococcus faecalis
    Phage length: 11,742 bp
    Has sequence: True

  Sample 4:
    Phage: Station25_SUR_ALL_assembly_NODE_3229_len
    Host: Bacillus cereus
    Phage length: 10,116 bp
    Has sequence: True

  Sample 5:
    Phage: Station30_DCM_ALL_assembly_NODE_1186_len
    Host: Prochlorococcus marinus
    Phage length: 16,368 bp
    Has sequence: True

✅ Streaming completed


In [17]:
# Example 12: Using datasets with PyTorch (if available)
print("Example 12: PyTorch Integration")
print("-" * 60)

try:
    from torch.utils.data import DataLoader
    from pbi import phage_host_collate_fn
    
    # Create indexed dataset (supports shuffling)
    dataset = retriever.create_indexed_dataset(
        where_clause="LIMIT 100"
    )
    
    # Create DataLoader with custom collate function
    dataloader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=True,  # Randomly shuffle samples
        num_workers=0,  # Set to 2-4 for parallel loading
        collate_fn=phage_host_collate_fn  # Handle mixed types
    )
    
    print(f"Created DataLoader:")
    print(f"  Dataset size: {len(dataset)}")
    print(f"  Batch size: 8")
    print(f"  Num batches: {len(dataloader)}")
    
    # Get one batch
    batch = next(iter(dataloader))
    print(f"\nBatch structure:")
    print(f"  Type: {type(batch)}")
    print(f"  Keys: {list(batch.keys())}")
    print(f"  Phage_IDs in batch: {len(batch['Phage_ID'])}")
    print(f"  Sample Phage_ID: {batch['Phage_ID'][0][:40]}")
    
except ImportError:
    print("⚠️  PyTorch not installed")
    print("   Install with: pip install torch")
    print("   Datasets will still work, but without DataLoader features")

2026-02-20 07:38:40,682 - INFO - Using host mapping with 5525 hosts


Example 12: PyTorch Integration
------------------------------------------------------------


2026-02-20 07:38:41,107 - INFO - Loaded metadata for 100 phage-host pairs


Created DataLoader:
  Dataset size: 100
  Batch size: 8
  Num batches: 13

Batch structure:
  Type: <class 'dict'>
  Keys: ['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence']
  Phage_IDs in batch: 8
  Sample Phage_ID: Station30_DCM_ALL_assembly_NODE_2287_len


### Context Manager for Cleanup

Both dataset types support Python's context manager protocol for automatic cleanup:

In [18]:
# Example 13: Using context manager (recommended)
print("Example 13: Context Manager (Automatic Cleanup)")
print("-" * 60)

# Using 'with' ensures files are closed properly
with retriever.create_indexed_dataset(where_clause="LIMIT 20") as dataset:
    print(f"Dataset opened: {len(dataset)} samples")
    
    # Use dataset
    sample = dataset[0]
    print(f"First sample: {sample['Phage_ID'][:40]}")
    
    # Files automatically closed when exiting 'with' block
print("✅ Dataset closed automatically")

# Alternative: Manual cleanup
dataset = retriever.create_indexed_dataset(where_clause="LIMIT 20")
try:
    print(f"\nDataset opened: {len(dataset)} samples")
finally:
    dataset.close()  # Explicit cleanup
    print("✅ Dataset closed manually")

2026-02-20 07:38:56,214 - INFO - Using host mapping with 5525 hosts


Example 13: Context Manager (Automatic Cleanup)
------------------------------------------------------------


2026-02-20 07:38:56,706 - INFO - Loaded metadata for 20 phage-host pairs


Dataset opened: 20 samples
First sample: Station23_DCM_ALL_assembly_NODE_1538_len


2026-02-20 07:39:11,335 - INFO - Using host mapping with 5525 hosts


✅ Dataset closed automatically


2026-02-20 07:39:11,776 - INFO - Loaded metadata for 20 phage-host pairs



Dataset opened: 20 samples
✅ Dataset closed manually


---

## 7. Custom Transformations

You can apply custom transformations to samples for preprocessing:

---

## 9. Multi-Host Support

Since phage Host fields can contain semicolon-separated values referencing **multiple host species**,
the pipeline now produces two additional CSV files during the host resolution step:

| File | Description |
|------|-------------|
| `phage_host_candidates.csv` | One row per (Phage_ID, token) – lossless record of every candidate parsed from the Host field |
| `phage_host_assemblies.csv` | One row per (Phage_ID, Assembly_Accession) link with rank, confidence, and provenance |

These files are tracked as Snakemake outputs, so the rule is **not re-run** when they already exist
(idempotent / cached execution).

### Host field parsing examples

```
Input:  'NA;GCA 900066335.1;UBA9502;Blautia obeum'
Tokens: GCA_900066335.1 (assembly_accession, order=2)
        UBA9502          (other,              order=3)
        Blautia obeum    (species_name,        order=4)

Input:  'Bacteroides dorei;Bacteroides vulgatus'
Tokens: Bacteroides dorei    (species_name, order=1)
        Bacteroides vulgatus (species_name, order=2)
```


In [35]:
# Example 14: Custom transformation
print("Example 14: Custom Transformation")
print("-" * 60)

def sequence_encoding_transform(sample):
    """Example: Convert sequences to one-hot encoding (simplified)"""
    # Add custom fields
    sample['Phage_Length_Category'] = (
        'Large' if sample['Phage_Length'] > 50000 else 'Small'
    )
    
    # You could add:
    # - One-hot encoding of sequences
    # - K-mer features
    # - GC content windows
    # - Etc.
    
    return sample

# Create dataset with transform
transformed_dataset = retriever.create_indexed_dataset(
    where_clause="LIMIT 10",
    transform=sequence_encoding_transform
)

# Check transformed sample
sample = transformed_dataset[0]
print("Transformed sample keys:")
print(list(sample.keys()))
print(f"\nNew field - Phage_Length_Category: {sample['Phage_Length_Category']}")
print(f"Original Phage_Length: {sample['Phage_Length']:,} bp")

2026-02-20 07:56:37,061 - INFO - Using host mapping with 5525 hosts


Example 14: Custom Transformation
------------------------------------------------------------


2026-02-20 07:56:37,459 - INFO - Loaded metadata for 10 phage-host pairs
2026-02-20 07:56:51,021 - WARNING - ⚠️  Host genome not in mapping (not downloaded/available for this host): GCF_979243125_1


Transformed sample keys:
['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence', 'Phage_Length_Category']

New field - Phage_Length_Category: Small
Original Phage_Length: 13,740 bp


In [ ]:
# Example 15: Explore multi-host candidate and assembly CSV outputs
import pandas as pd
from pathlib import Path

print("Multi-Host Support – Pipeline Output Files")
print("=" * 60)

# Adjust paths to match your PBI_DATA_DIR setting
candidates_path = Path("/data/intermediate/csv/merged/phage_host_candidates.csv")
assemblies_path = Path("/data/intermediate/csv/merged/phage_host_assemblies.csv")

if candidates_path.exists():
    cand_df = pd.read_csv(candidates_path)
    print(f"\nphage_host_candidates.csv: {len(cand_df):,} rows")
    print(cand_df.head(5))
    print(f"\nMulti-host phages (>1 candidate):")
    multi = cand_df.groupby('Phage_ID').size()
    print(multi[multi > 1].head(5))
else:
    print(f"⚠️  {candidates_path} not found – run the pipeline first.")
    # Demonstrate parsing without running the pipeline
    import sys, os
    sys.path.insert(0, str(Path(os.getcwd()) / 'workflow' / 'scripts' / 'sequences'))
    from download_host_genomes_robust import parse_host_field
    examples = [
        'NA;GCA 900066335.1;UBA9502;Blautia obeum',
        'Bacteroides dorei;Bacteroides vulgatus',
        '-',
    ]
    for ex in examples:
        tokens = parse_host_field(ex)
        print(f"\nHost: {ex!r}")
        for t in tokens:
            print(f"  [{t.token_order}] {t.token!r:30s}  ({t.token_type})")
        if not tokens:
            print("  (no valid tokens)")

if assemblies_path.exists():
    asm_df = pd.read_csv(assemblies_path)
    print(f"\nphage_host_assemblies.csv: {len(asm_df):,} rows")
    print(asm_df[['Phage_ID', 'Host_Token', 'Assembly_Accession',
                   'Resolution_Source', 'Confidence']].head(5))
else:
    print(f"\n⚠️  {assemblies_path} not found – run the pipeline first.")


Multi-Host Support – Pipeline Output Files

phage_host_candidates.csv: 879,615 rows
                                            Phage_ID  \
0  6982.1.58137.GGACC_Malaspina_NODE_113_length_3...   
1  6982.1.58137.GGACC_Malaspina_NODE_215_length_2...   
2  6982.1.58137.GGACC_Malaspina_NODE_229_length_2...   
3  6982.1.58137.GGACC_Malaspina_NODE_267_length_2...   
4  6982.1.58137.GGACC_Malaspina_NODE_486_length_1...   

                    Host_Raw                 Host_Token    Token_Type  \
0   Lawsonia intracellularis   Lawsonia intracellularis  species_name   
1  Mycobacteroides abscessus  Mycobacteroides abscessus  species_name   
2       Pseudomonas syringae       Pseudomonas syringae  species_name   
3   Lawsonia intracellularis   Lawsonia intracellularis  species_name   
4       Enterobacter cloacae       Enterobacter cloacae  species_name   

   Token_Order  
0            1  
1            1  
2            1  
3            1  
4            1  

Multi-host phages (>1 candidate):
Pha

---

## 8. Summary and Best Practices

### What We Covered

✅ **Database Connection**: Quick setup with auto-detection  
✅ **Data Filtering**: Flexible SQL-based queries  
✅ **Sequence Retrieval**: Both non-streaming and streaming approaches  
✅ **Log Analysis**: Understanding downloads and failures  
✅ **ML Integration**: PyTorch-compatible datasets  
✅ **Resource Management**: Context managers and cleanup  

### Best Practices

#### For Data Exploration (Notebooks)
- Use `get_phage_host_pairs()` for small queries (<10k pairs)
- Use `LIMIT` to preview data before full queries
- Check metadata CSVs before working with sequences

#### For Machine Learning
- Use **IndexedDataset** for medium datasets with shuffling
- Use **StreamingDataset** for large datasets (>100k pairs)
- Always use `with` context manager or call `close()`
- Use `missing_hosts_csv` to track data quality

#### For Production
- Monitor failure logs during downloads
- Validate `Sequence_Count` for quality control
- Archive logs with dates for reproducibility
- Use NCBI API key for faster downloads

### Next Steps

📖 **Documentation:**
- [Pipeline Execution Guide](../docs/guides/pipeline-execution.md) - Run the full pipeline
- [Analysis Guide](../docs/guides/analysis-guide.md) - Advanced queries
- [Machine Learning Guide](../docs/guides/machine-learning.md) - Training models

🔬 **Example Notebooks:**
- `example_streaming_ml.ipynb` - Full ML workflow
- `ml_1_phage_host_dataset.ipynb` - Dataset details
- `analysis_direct_access_guide.ipynb` - Database queries

### Getting Help

```python
# Built-in help
retriever.help()

# Package documentation
help(pbi.SequenceRetriever)
```

Happy analyzing! 🦠🔬